# Intro to CI/CD: core concepts with interactive examples

> last_verified: 2026-07-23 · n/a (concept)

I wanted to understand CI/CD beyond the definitions — so I worked through pipeline stages and approval gates with real-ish examples. This notebook captures what I tried, what broke, and what clicked.

Source: [CI/CD exercises — Times of Cloud](https://timesofcloud.com/gcp/devops/cicd-exercises/)

## What I'm trying to understand

- **CI (Continuous Integration):** merge early, merge often, and run automated checks on every change.
- **CD (Continuous Delivery / Deployment):** get every passing change into a releasable state.
- **Pipeline stages:** the sequence of steps (lint -> build -> test -> deploy) that code goes through.
- **Gates:** manual approvals or automated checks that block a stage from proceeding.

## Step 1: A minimal CI pipeline

I wrote a shell script that simulates what a real CI runner does: check out code, lint, build, and test. It's the same shape as a GitHub Actions workflow but without the YAML.

In [ ]:
%%bash

# Simulating what a CI pipeline runner does

PROJECT_DIR=$(mktemp -d)
trap 'rm -rf "$PROJECT_DIR"' EXIT
cd "$PROJECT_DIR" || exit 1

# Set up a fake project
cat > app.py <<'PY'
def add(a, b):
    return a + b
if __name__ == "__main__":
    print(f"add(2,3) = {add(2,3)}")
PY

cat > test_app.py <<'PY'
from app import add
assert add(2, 3) == 5
assert add(-1, 1) == 0
print("All tests passed")
PY

echo "=== CI Pipeline ==="

# 1. Lint
python3 -m py_compile app.py && echo "  Lint: app.py OK"
python3 -m py_compile test_app.py && echo "  Lint: test_app.py OK"

# 2. Build (interpreted - just verify files exist)
test -f app.py && test -f test_app.py && echo "  Build: source files present"

# 3. Test
python3 test_app.py && echo "  Test: passed" && echo "\nPipeline: SUCCESS"

## Step 2: Adding a deploy step and approval gate

After CI passes, the code needs to get to environments. Dev environments deploy automatically. Staging needs a manual approval before changes go further.

I simulated this with a two-stage deploy and a prompt-based gate.

In [ ]:
%%bash

# Simulating multi-stage deployment with an approval gate

echo "=== CD Pipeline ==="

# Stage: Deploy to dev (auto)
echo "  Deploying to dev... (auto)"
echo "  Dev deploy complete"

# Stage: Deploy to staging (requires CI green)
echo "  Deploying to staging... (auto after CI)"
echo "  Staging deploy complete - running smoke test"

# Smoke test
echo "  Smoke test: PASS"

# Stage: Approval gate before promoting further
echo ""
echo "  === Approval Gate ==="
echo "  Type 'yes' to promote staging to release, anything else to skip:"
read -r APPROVAL

if [ "$APPROVAL" = "yes" ]; then
    echo "  Promoting staging release..."
    echo "  Release complete"
else
    echo "  Release SKIPPED (awaiting approval)"
fi

echo "\nPipeline: COMPLETE"

## What I got stuck on

- **Approval gates:** I kept trying to write `if: always()` when I meant `if: success()` for conditional stage triggers. A stage should only run if its prerequisite passed.
- **`latest` tag trap:** I initially tagged everything `latest` and couldn't tell which version was deployed where. Using unique identifiers (like the git commit SHA) for each build fixed this.
- **Hardcoded secrets:** my first draft had credentials in the YAML. I learned to use environment variables or a secrets manager instead.

## What I'd try next

I want to wire this up in a real GitHub Actions workflow with separate environments and a manual approval gate between dev and staging. Then test the full cycle with an actual app build and deployment to see the pipeline end to end.